In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.4 Transformer block

Now stack the pieces into the real PocketLM block: pre-norm + causal attention +
residual, then pre-norm + MLP + residual. The whole model is just N of these in
a row.

In [ ]:
from pocketlm.model import Block, PocketLMConfig

torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=11, block_size=8, n_layer=1, n_head=2, n_embd=16)
block = Block(cfg).eval()
x = torch.randn(1, 5, 16)
out = block(x)
print("block in", tuple(x.shape), "-> out", tuple(out.shape),
      "(shape preserved, so blocks stack)")

Because the block preserves shape, you can stack any depth. Depth is where
abstraction comes from: early blocks see characters, later ones see structure.

In [ ]:
assert tuple(out.shape) == (1, 5, 16)